# Full Unsloth-Style Kernel Tests

Tests correctness, speed, and memory for all new fused kernels:
1. **Fused SiLU*Mul** — Triton kernel replacing `F.silu(gate) * up`
2. **Fused RoPE** — Triton kernel replacing PyTorch stack+reshape RoPE
3. **Fused SwiGLU** — autograd.Function fusing gate_proj + up_proj + silu_mul

**Requirements:** torch 2.7.1+cu128 (avoids 2.10 memory leak), triton (pulled by torch). On Colab:
```bash
pip install torch==2.7.1+cu128 torchvision==0.22.1+cu128 --index-url https://download.pytorch.org/whl/cu128
pip uninstall torchaudio -y
```
Then **restart runtime**. (Uninstalling torchaudio removes the conflict with torch 2.7.1; we don't need it for training.)

In [ ]:
# Optional: install pinned torch on Colab (avoids PyTorch 2.10 memory leak)
# !pip install torch==2.7.1+cu128 torchvision==0.22.1+cu128 --index-url https://download.pytorch.org/whl/cu128
# !pip uninstall torchaudio -y   # resolves "torchaudio requires torch==2.10" conflict
# Then restart runtime and run the rest.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

_torch_ver = getattr(torch, '__version__', '0.0.0')
if _torch_ver.startswith('2.10'):
    raise RuntimeError(f'PyTorch {_torch_ver} has a memory leak. Use torch 2.7.1+cu128.')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    p = torch.cuda.get_device_properties(0)
    mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9
    print(f'VRAM: {mem:.1f} GB')

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
    print(f'Triton: {triton.__version__}')
except ImportError:
    HAS_TRITON = False
    print('Triton not available')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16

## 1. Fused SiLU*Mul Kernel

In [ ]:
# ── Triton Fused SiLU*Mul ──

if HAS_TRITON:
    @triton.jit
    def _silu_mul_fwd_kernel(OUT_ptr, GATE_ptr, UP_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        gate_raw = tl.load(GATE_ptr + offsets, mask=mask)
        up_raw = tl.load(UP_ptr + offsets, mask=mask)
        out_dtype = gate_raw.dtype
        gate = gate_raw.to(tl.float32)
        up = up_raw.to(tl.float32)
        sigma = tl.sigmoid(gate)
        result = gate * sigma * up
        tl.store(OUT_ptr + offsets, result.to(out_dtype), mask=mask)

    @triton.jit
    def _silu_mul_bwd_kernel(D_GATE_ptr, D_UP_ptr, GRAD_ptr, GATE_ptr, UP_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        gate_raw = tl.load(GATE_ptr + offsets, mask=mask)
        out_dtype = gate_raw.dtype
        grad = tl.load(GRAD_ptr + offsets, mask=mask).to(tl.float32)
        gate = gate_raw.to(tl.float32)
        up = tl.load(UP_ptr + offsets, mask=mask).to(tl.float32)
        sigma = tl.sigmoid(gate)
        silu_gate = gate * sigma
        d_up = silu_gate * grad
        d_gate = sigma * (1.0 + gate * (1.0 - sigma)) * up * grad
        tl.store(D_GATE_ptr + offsets, d_gate.to(out_dtype), mask=mask)
        tl.store(D_UP_ptr + offsets, d_up.to(out_dtype), mask=mask)


class _FusedSiLUMulFunction(torch.autograd.Function):
    @staticmethod
    @torch.amp.custom_fwd(device_type='cuda')
    def forward(ctx, gate, up):
        out = torch.empty_like(gate)
        n = gate.numel()
        BLOCK = 1024
        _silu_mul_fwd_kernel[(n + BLOCK - 1) // BLOCK,](out, gate, up, n, BLOCK_SIZE=BLOCK)
        ctx.save_for_backward(gate, up)
        return out

    @staticmethod
    @torch.amp.custom_bwd(device_type='cuda')
    def backward(ctx, grad_output):
        gate, up = ctx.saved_tensors
        grad_output = grad_output.contiguous()
        d_gate = torch.empty_like(gate)
        d_up = torch.empty_like(up)
        n = gate.numel()
        BLOCK = 1024
        _silu_mul_bwd_kernel[(n + BLOCK - 1) // BLOCK,](d_gate, d_up, grad_output, gate, up, n, BLOCK_SIZE=BLOCK)
        return d_gate, d_up


def fused_silu_mul(gate, up):
    if HAS_TRITON and gate.is_cuda and gate.is_contiguous() and up.is_contiguous():
        return _FusedSiLUMulFunction.apply(gate, up)
    return F.silu(gate) * up


def standard_silu_mul(gate, up):
    return F.silu(gate) * up


print('SiLU*Mul kernel defined')

## 2. Fused RoPE Kernel

In [ ]:
# ── Triton Fused RoPE ──

if HAS_TRITON:
    @triton.jit
    def _rope_fwd_kernel(OUT_ptr, X_ptr, COS_ptr, SIN_ptr, n_rows, half_dim,
                          stride_x_row, stride_out_row, stride_cos_row, BLOCK_HALF: tl.constexpr):
        row_id = tl.program_id(0)
        offs = tl.arange(0, BLOCK_HALF)
        mask = offs < half_dim
        x_base = row_id * stride_x_row
        cos_base = row_id * stride_cos_row
        out_base = row_id * stride_out_row
        x_even_raw = tl.load(X_ptr + x_base + offs * 2, mask=mask)
        out_dtype = x_even_raw.dtype
        x_even = x_even_raw.to(tl.float32)
        x_odd = tl.load(X_ptr + x_base + offs * 2 + 1, mask=mask).to(tl.float32)
        cos_val = tl.load(COS_ptr + cos_base + offs, mask=mask).to(tl.float32)
        sin_val = tl.load(SIN_ptr + cos_base + offs, mask=mask).to(tl.float32)
        out_even = x_even * cos_val - x_odd * sin_val
        out_odd = x_even * sin_val + x_odd * cos_val
        tl.store(OUT_ptr + out_base + offs * 2, out_even.to(out_dtype), mask=mask)
        tl.store(OUT_ptr + out_base + offs * 2 + 1, out_odd.to(out_dtype), mask=mask)

    @triton.jit
    def _rope_bwd_kernel(D_X_ptr, GRAD_ptr, COS_ptr, SIN_ptr, n_rows, half_dim,
                          stride_grad_row, stride_dx_row, stride_cos_row, BLOCK_HALF: tl.constexpr):
        row_id = tl.program_id(0)
        offs = tl.arange(0, BLOCK_HALF)
        mask = offs < half_dim
        grad_base = row_id * stride_grad_row
        cos_base = row_id * stride_cos_row
        dx_base = row_id * stride_dx_row
        g_even_raw = tl.load(GRAD_ptr + grad_base + offs * 2, mask=mask)
        out_dtype = g_even_raw.dtype
        g_even = g_even_raw.to(tl.float32)
        g_odd = tl.load(GRAD_ptr + grad_base + offs * 2 + 1, mask=mask).to(tl.float32)
        cos_val = tl.load(COS_ptr + cos_base + offs, mask=mask).to(tl.float32)
        sin_val = tl.load(SIN_ptr + cos_base + offs, mask=mask).to(tl.float32)
        dx_even = g_even * cos_val + g_odd * sin_val
        dx_odd = -g_even * sin_val + g_odd * cos_val
        tl.store(D_X_ptr + dx_base + offs * 2, dx_even.to(out_dtype), mask=mask)
        tl.store(D_X_ptr + dx_base + offs * 2 + 1, dx_odd.to(out_dtype), mask=mask)


class _FusedRoPEFunction(torch.autograd.Function):
    @staticmethod
    @torch.amp.custom_fwd(device_type='cuda')
    def forward(ctx, x, cos, sin):
        full_dim = x.size(-1)
        half_dim = full_dim // 2
        orig_shape = x.shape
        x_flat = x.contiguous().view(-1, full_dim)
        n_rows = x_flat.shape[0]
        cos_flat = cos[..., :half_dim].contiguous().view(-1, half_dim)
        sin_flat = sin[..., :half_dim].contiguous().view(-1, half_dim)
        if cos_flat.shape[0] != n_rows:
            cos_flat = cos_flat.repeat(n_rows // cos_flat.shape[0], 1)
            sin_flat = sin_flat.repeat(n_rows // sin_flat.shape[0], 1)
        out = torch.empty_like(x_flat)
        BLOCK_HALF = triton.next_power_of_2(half_dim)
        _rope_fwd_kernel[(n_rows,)](out, x_flat, cos_flat, sin_flat, n_rows, half_dim,
                                     x_flat.stride(0), out.stride(0), cos_flat.stride(0), BLOCK_HALF=BLOCK_HALF)
        ctx.save_for_backward(cos_flat, sin_flat)
        ctx.orig_shape = orig_shape
        return out.view(orig_shape)

    @staticmethod
    @torch.amp.custom_bwd(device_type='cuda')
    def backward(ctx, grad_output):
        cos_flat, sin_flat = ctx.saved_tensors
        orig_shape = ctx.orig_shape
        full_dim = orig_shape[-1]
        half_dim = full_dim // 2
        grad_flat = grad_output.contiguous().view(-1, full_dim)
        n_rows = grad_flat.shape[0]
        dx = torch.empty_like(grad_flat)
        BLOCK_HALF = triton.next_power_of_2(half_dim)
        _rope_bwd_kernel[(n_rows,)](dx, grad_flat, cos_flat, sin_flat, n_rows, half_dim,
                                     grad_flat.stride(0), dx.stride(0), cos_flat.stride(0), BLOCK_HALF=BLOCK_HALF)
        return dx.view(orig_shape), None, None


def fused_rope(x, cos, sin):
    if HAS_TRITON and x.is_cuda and x.is_contiguous():
        return _FusedRoPEFunction.apply(x, cos, sin)
    return standard_rope(x, cos, sin)


def standard_rope(x, cos, sin):
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    cos_half = cos[..., :x.size(-1) // 2]
    sin_half = sin[..., :x.size(-1) // 2]
    rot_even = x_even * cos_half - x_odd * sin_half
    rot_odd = x_even * sin_half + x_odd * cos_half
    return torch.stack((rot_even, rot_odd), dim=-1).reshape_as(x)


print('RoPE kernel defined')

## 3. Fused SwiGLU (gate + up + silu_mul)

In [ ]:
# ── Fused SwiGLU autograd ──

class _FusedSwiGLUFunc(torch.autograd.Function):
    @staticmethod
    @torch.amp.custom_fwd(device_type='cuda')
    def forward(ctx, x, W_gate, W_up):
        gate_h = F.linear(x, W_gate)
        up_h = F.linear(x, W_up)
        if HAS_TRITON and gate_h.is_cuda:
            h = torch.empty_like(gate_h)
            n = gate_h.numel()
            BLOCK = 1024
            _silu_mul_fwd_kernel[(n + BLOCK - 1) // BLOCK,](
                h, gate_h.reshape(-1), up_h.reshape(-1), n, BLOCK_SIZE=BLOCK)
        else:
            h = F.silu(gate_h) * up_h
        ctx.save_for_backward(x, W_gate, W_up)
        return h

    @staticmethod
    @torch.amp.custom_bwd(device_type='cuda')
    def backward(ctx, grad_h):
        x, W_gate, W_up = ctx.saved_tensors
        orig_shape = x.shape
        x_2d = x.reshape(-1, x.shape[-1])
        go_2d = grad_h.reshape(-1, grad_h.shape[-1])
        gate_h = x_2d @ W_gate.t()
        up_h = x_2d @ W_up.t()
        if HAS_TRITON and gate_h.is_cuda:
            n = gate_h.numel()
            d_gate_h = torch.empty_like(gate_h)
            d_up_h = torch.empty_like(up_h)
            BLOCK = 1024
            _silu_mul_bwd_kernel[(n + BLOCK - 1) // BLOCK,](
                d_gate_h.reshape(-1), d_up_h.reshape(-1),
                go_2d.contiguous().reshape(-1),
                gate_h.reshape(-1), up_h.reshape(-1),
                n, BLOCK_SIZE=BLOCK)
            del gate_h, up_h
        else:
            sigma = torch.sigmoid(gate_h)
            silu_gate = gate_h * sigma
            d_up_h = silu_gate * go_2d
            d_gate_h = sigma * (1.0 + gate_h * (1.0 - sigma)) * up_h * go_2d
            del gate_h, up_h, sigma, silu_gate
        grad_W_gate = d_gate_h.t() @ x_2d
        grad_W_up = d_up_h.t() @ x_2d
        grad_x_2d = d_gate_h @ W_gate
        grad_x_2d.addmm_(d_up_h, W_up)
        del d_gate_h, d_up_h
        return grad_x_2d.reshape(orig_shape), grad_W_gate, grad_W_up


class FusedSwiGLU(nn.Module):
    def __init__(self, in_f, hidden_f, dtype=torch.bfloat16):
        super().__init__()
        self.gate_proj = nn.Linear(in_f, hidden_f, bias=False, dtype=dtype)
        self.up_proj = nn.Linear(in_f, hidden_f, bias=False, dtype=dtype)
    def forward(self, x):
        return _FusedSwiGLUFunc.apply(x, self.gate_proj.weight, self.up_proj.weight)


class StandardSwiGLU(nn.Module):
    def __init__(self, in_f, hidden_f, dtype=torch.bfloat16):
        super().__init__()
        self.gate_proj = nn.Linear(in_f, hidden_f, bias=False, dtype=dtype)
        self.up_proj = nn.Linear(in_f, hidden_f, bias=False, dtype=dtype)
    def forward(self, x):
        return F.silu(self.gate_proj(x)) * self.up_proj(x)


print('SwiGLU modules defined')

## 4. Correctness Tests

**Note on bf16 thresholds:** The fused Triton kernels (SiLU\*Mul, RoPE) compute in fp32
internally and round once at the end, so elementwise diffs are small (~0.06).
SwiGLU recomputes `gate_h`/`up_h` in backward via `x @ W.t()` instead of using the saved
tensors from `F.linear`, which can take a different matmul codepath — so weight grad diffs
can be larger (~0.5–1.0) but are within bf16 precision.

In [ ]:
# bf16 tolerance: fused (fp32 internal) vs standard (bf16 intermediates)
# Elementwise ops (SiLU*Mul, RoPE): differ by up to ~0.06 (1 ULP at tail values)
# SwiGLU grads: recomputed gate_h/up_h may differ from saved originals due to
#   different matmul codepaths (F.linear on 3D vs manual reshape + @ on 2D).
FWD_TOL = 0.1        # elementwise forward
BWD_TOL = 0.1        # elementwise backward
SWIGLU_X_TOL = 0.5   # SwiGLU grad_x (recompute + matmul chain)
SWIGLU_W_TOL = 1.0   # SwiGLU weight grads (bf16 matmul accumulation differs)

print('=' * 60)
print('CORRECTNESS TESTS')
print('=' * 60)

torch.manual_seed(42)

# ── SiLU*Mul ──
print('\n--- SiLU*Mul ---')
gate = torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True)
up = torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True)
gate2 = gate.detach().clone().requires_grad_(True)
up2 = up.detach().clone().requires_grad_(True)

out_std = standard_silu_mul(gate, up)
out_fused = fused_silu_mul(gate2, up2)
fwd_diff = (out_std - out_fused).abs().max().item()
print(f'  Forward max_diff: {fwd_diff:.6e} [{"PASS" if fwd_diff < FWD_TOL else "FAIL"}]')

out_std.sum().backward()
out_fused.sum().backward()
dg_diff = (gate.grad - gate2.grad).abs().max().item()
du_diff = (up.grad - up2.grad).abs().max().item()
print(f'  grad_gate max_diff: {dg_diff:.6e} [{"PASS" if dg_diff < BWD_TOL else "FAIL"}]')
print(f'  grad_up max_diff:   {du_diff:.6e} [{"PASS" if du_diff < BWD_TOL else "FAIL"}]')

del gate, up, gate2, up2, out_std, out_fused
torch.cuda.empty_cache()

# ── RoPE ──
print('\n--- RoPE ---')
B, T, H, D = 2, 256, 8, 64
x = torch.randn(B, T, H, D, dtype=DTYPE, device=DEVICE, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)
cos = torch.randn(T, D // 2, dtype=DTYPE, device=DEVICE)
sin = torch.randn(T, D // 2, dtype=DTYPE, device=DEVICE)
cos_b = cos.unsqueeze(0).unsqueeze(2).expand(B, T, H, D // 2)
sin_b = sin.unsqueeze(0).unsqueeze(2).expand(B, T, H, D // 2)

out_std = standard_rope(x, cos_b, sin_b)
out_fused = fused_rope(x2, cos_b, sin_b)
fwd_diff = (out_std - out_fused).abs().max().item()
print(f'  Forward max_diff: {fwd_diff:.6e} [{"PASS" if fwd_diff < FWD_TOL else "FAIL"}]')

out_std.sum().backward()
out_fused.sum().backward()
dx_diff = (x.grad - x2.grad).abs().max().item()
print(f'  grad_x max_diff:  {dx_diff:.6e} [{"PASS" if dx_diff < BWD_TOL else "FAIL"}]')

del x, x2, cos, sin, cos_b, sin_b, out_std, out_fused
torch.cuda.empty_cache()

# ── SwiGLU ──
print('\n--- SwiGLU ---')
in_f, hidden_f = 1024, 2048
torch.manual_seed(42)
std_m = StandardSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
fused_m = FusedSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
fused_m.gate_proj.weight.data.copy_(std_m.gate_proj.weight.data)
fused_m.up_proj.weight.data.copy_(std_m.up_proj.weight.data)

x = torch.randn(2, 256, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)

out_std = std_m(x)
out_fused = fused_m(x2)
fwd_diff = (out_std - out_fused).abs().max().item()
print(f'  Forward max_diff: {fwd_diff:.6e} [{"PASS" if fwd_diff < FWD_TOL else "FAIL"}]')

out_std.sum().backward()
out_fused.sum().backward()
dx_diff = (x.grad - x2.grad).abs().max().item()
dg_diff = (std_m.gate_proj.weight.grad - fused_m.gate_proj.weight.grad).abs().max().item()
du_diff = (std_m.up_proj.weight.grad - fused_m.up_proj.weight.grad).abs().max().item()
print(f'  grad_x max_diff:    {dx_diff:.6e} [{"PASS" if dx_diff < SWIGLU_X_TOL else "FAIL"}]')
print(f'  grad_gate max_diff: {dg_diff:.6e} [{"PASS" if dg_diff < SWIGLU_W_TOL else "FAIL"}]')
print(f'  grad_up max_diff:   {du_diff:.6e} [{"PASS" if du_diff < SWIGLU_W_TOL else "FAIL"}]')

del std_m, fused_m, x, x2, out_std, out_fused
torch.cuda.empty_cache()

print('\nAll correctness tests done.')

## 5. Speed Benchmarks

In [ ]:
def bench_speed(fn, args, label, n_warmup=10, n_runs=50):
    for _ in range(n_warmup):
        out = fn(*args)
        if out.requires_grad:
            out.sum().backward()
        elif isinstance(out, tuple):
            sum(o.sum() for o in out).backward()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_runs):
        out = fn(*args)
        if hasattr(out, 'sum'):
            out.sum().backward()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end) / n_runs
    print(f'  {label}: {ms:.3f} ms/iter')
    return ms


print('=' * 60)
print('SPEED BENCHMARKS')
print('=' * 60)

# ── SiLU*Mul ──
print('\n--- SiLU*Mul (4, 512, 2048) ---')
gate = torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True)
up = torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True)
gate2 = gate.detach().clone().requires_grad_(True)
up2 = up.detach().clone().requires_grad_(True)
t_std = bench_speed(standard_silu_mul, (gate, up), 'Standard')
t_fused = bench_speed(fused_silu_mul, (gate2, up2), 'Fused')
print(f'  Speedup: {t_std / t_fused:.2f}x')
del gate, up, gate2, up2
torch.cuda.empty_cache()

# ── RoPE ──
print('\n--- RoPE (2, 512, 8, 64) ---')
B, T, H, D = 2, 512, 8, 64
x = torch.randn(B, T, H, D, dtype=DTYPE, device=DEVICE, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)
cos = torch.randn(B, T, H, D // 2, dtype=DTYPE, device=DEVICE)
sin = torch.randn(B, T, H, D // 2, dtype=DTYPE, device=DEVICE)
t_std = bench_speed(standard_rope, (x, cos, sin), 'Standard')
t_fused = bench_speed(fused_rope, (x2, cos, sin), 'Fused')
print(f'  Speedup: {t_std / t_fused:.2f}x')
del x, x2, cos, sin
torch.cuda.empty_cache()

# ── SwiGLU ──
print('\n--- SwiGLU (2, 512, 1024 -> 2048) ---')
in_f, hidden_f = 1024, 2048
torch.manual_seed(42)
std_m = StandardSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
fused_m = FusedSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
fused_m.gate_proj.weight.data.copy_(std_m.gate_proj.weight.data)
fused_m.up_proj.weight.data.copy_(std_m.up_proj.weight.data)
x = torch.randn(2, 512, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)
t_std = bench_speed(std_m, (x,), 'Standard')
t_fused = bench_speed(fused_m, (x2,), 'Fused')
print(f'  Speedup: {t_std / t_fused:.2f}x')
del std_m, fused_m, x, x2
torch.cuda.empty_cache()

## 6. Memory Benchmarks

In [ ]:
import gc

def bench_mem_isolated(make_fn, make_args, label, n_warmup=2, n_runs=3):
    """Run a memory benchmark in complete isolation.
    make_fn: callable() -> function to benchmark
    make_args: callable() -> tuple of args (fresh each time to avoid cross-test pollution)
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    fn = make_fn()
    args = make_args()

    for _ in range(n_warmup):
        out = fn(*args)
        if hasattr(out, 'sum'):
            out.sum().backward()
        # Zero grads to avoid accumulation
        for a in args:
            if hasattr(a, 'grad') and a.grad is not None:
                a.grad = None
        torch.cuda.synchronize()

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e6

    for _ in range(n_runs):
        out = fn(*args)
        if hasattr(out, 'sum'):
            out.sum().backward()
        for a in args:
            if hasattr(a, 'grad') and a.grad is not None:
                a.grad = None
        torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated() / 1e6
    print(f'  {label}: peak={peak:.1f} MB, before={mem_before:.1f} MB')

    del fn, args, out
    gc.collect()
    torch.cuda.empty_cache()
    return peak


print('=' * 60)
print('MEMORY BENCHMARKS')
print('=' * 60)

# ── SiLU*Mul ──
print('\n--- SiLU*Mul (4, 512, 2048) ---')
def _make_silu_args():
    return (torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True),
            torch.randn(4, 512, 2048, dtype=DTYPE, device=DEVICE, requires_grad=True))
p_std = bench_mem_isolated(lambda: standard_silu_mul, _make_silu_args, 'Standard')
p_fused = bench_mem_isolated(lambda: fused_silu_mul, _make_silu_args, 'Fused')
print(f'  Savings: {(p_std - p_fused) / p_std * 100:.1f}% ({p_std - p_fused:.1f} MB)')

# ── SwiGLU ──
print('\n--- SwiGLU (2, 512, 1024 -> 2048) ---')
in_f, hidden_f = 1024, 2048
_swiglu_W_gate = torch.randn(hidden_f, in_f, dtype=DTYPE, device=DEVICE)
_swiglu_W_up = torch.randn(hidden_f, in_f, dtype=DTYPE, device=DEVICE)

def _make_std_swiglu():
    m = StandardSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
    m.gate_proj.weight.data.copy_(_swiglu_W_gate)
    m.up_proj.weight.data.copy_(_swiglu_W_up)
    return m

def _make_fused_swiglu():
    m = FusedSwiGLU(in_f, hidden_f, dtype=DTYPE).to(DEVICE)
    m.gate_proj.weight.data.copy_(_swiglu_W_gate)
    m.up_proj.weight.data.copy_(_swiglu_W_up)
    return m

def _make_swiglu_args():
    return (torch.randn(2, 512, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True),)

p_std = bench_mem_isolated(_make_std_swiglu, _make_swiglu_args, 'Standard')
p_fused = bench_mem_isolated(_make_fused_swiglu, _make_swiglu_args, 'Fused')
print(f'  Savings: {(p_std - p_fused) / p_std * 100:.1f}% ({p_std - p_fused:.1f} MB)')

del _swiglu_W_gate, _swiglu_W_up
gc.collect()
torch.cuda.empty_cache()

## 7. Summary

In [ ]:
print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print()
print('Unsloth-style fused kernels for 70B MoE training:')
print()
print('1. Fused SiLU*Mul (Triton):')
print('   Replaces F.silu(gate) * up with single Triton kernel.')
print('   ~3x less memory bandwidth, fused backward.')
print('   Used in: MoE shared expert + routed experts (all 20 layers).')
print()
print('2. Fused RoPE (Triton):')
print('   Replaces torch.stack + reshape with in-place rotation.')
print('   No intermediate tensors.')
print('   Used in: Every attention layer (20 layers).')
print()
print('3. Fused SwiGLU (autograd.Function):')
print('   gate_proj + up_proj + silu_mul in one autograd node.')
print('   Recomputes gate_h/up_h in backward (Unsloth philosophy).')
print('   Eliminates 2 full [B*T, d_hidden] activation tensors per layer.')
print('   Used in: MoE shared expert path (all 20 layers).')
print()
print('4. Fused DeltaNet Entrance (already implemented):')
print('   conv + L2Norm + RoPE in one Triton kernel.')
print('   Now enabled by default (was off).')
print('   Used in: 15 DeltaNet layers.')
print()
print('Combined with previously added FusedLoRALinear,')
print('this covers the full Unsloth philosophy across the 70B hot path.')
print()
print('Note on bf16 precision:')
print('  Triton elementwise kernels compute in fp32 internally, rounding once.')
print('  SwiGLU backward recomputes gate_h/up_h in native dtype (bf16).')
print('  Weight grad diffs up to ~1.0 are expected (different matmul codepaths).')
print('  Elementwise diffs up to ~0.1 are expected and correct.')